# Skeletons

**Geometry type:** `skeleton`

Trees of vertices joined by parent links, as in SWC neuron morphologies. Each
tree is one object.

1. Generate synthetic neuron morphologies
2. Write one skeleton with `write_graph(kind="skeleton")`
3. Write several skeletons into one store with `object_ids`
4. Convert an SWC file and export it back
5. Build a pyramid and measure each tree at every level
6. Validate

In [1]:
import os
import tempfile

import numpy as np
import pandas as pd
import zarr_vectors as zv
from zarr_vectors.types.graphs import read_graph, write_graph
from zarr_vectors.validate import validate

pd.set_option("display.width", 120)

_tmpdir = tempfile.mkdtemp(prefix="zv_examples_")
STORE       = os.path.join(_tmpdir, "neuron.zv")
STORE_MULTI = os.path.join(_tmpdir, "neurons.zv")
SWC_IN      = os.path.join(_tmpdir, "neurons.swc")
STORE_SWC   = os.path.join(_tmpdir, "neurons_from_swc.zv")
print("Working directory:", _tmpdir)

Working directory: /tmp/zv_examples_fgdg_cl2


## 1 · Generate synthetic neurons

Each neuron is a soma with 40 branches of 20–60 nodes, 2 µm apart. Each
branch starts at a random existing node of the soma or of its own
compartment: the first 10 branches form the axon (SWC type 2), the rest
basal dendrites (type 3).

In [2]:
rng = np.random.default_rng(3)

def generate_neuron(origin, n_branches=40, n_axon=10, step=2.0):
    pos, parent, swc_type = [np.asarray(origin, float)], [-1], [1]   # soma
    for b in range(n_branches):
        kind = 2 if b < n_axon else 3
        candidates = [i for i, t in enumerate(swc_type) if t in (1, kind)]
        node = candidates[int(rng.integers(len(candidates)))]
        direction = rng.normal(size=3)
        for _ in range(int(rng.integers(20, 60))):
            direction += rng.normal(0, 0.3, 3)
            direction /= np.linalg.norm(direction)
            pos.append(pos[node] + step * direction)
            parent.append(node)
            swc_type.append(kind)
            node = len(pos) - 1
    swc_type = np.array(swc_type, dtype=np.int32)
    radius = np.where(swc_type == 1, 10.0, rng.uniform(0.3, 2.0, len(pos))).astype(np.float32)
    return np.array(pos, dtype=np.float32), np.array(parent), swc_type, radius

neurons = [generate_neuron([200.0 + 150 * i, 300.0, 300.0]) for i in range(3)]
for i, (pos, parent, swc_type, radius) in enumerate(neurons):
    types, counts = np.unique(swc_type, return_counts=True)
    print(f"neuron {i}: {len(pos):,} nodes, nodes per SWC type {dict(zip(types.tolist(), counts.tolist()))}")

neuron 0: 1,618 nodes, nodes per SWC type {1: 1, 2: 412, 3: 1205}
neuron 1: 1,550 nodes, nodes per SWC type {1: 1, 2: 403, 3: 1146}
neuron 2: 1,648 nodes, nodes per SWC type {1: 1, 2: 386, 3: 1261}


## 2 · Write one skeleton

`edges` are `[child, parent]` pairs. The store keeps a parent link implied by
node order where it can and stores the rest as explicit links, so the stored
link counts in the summary are far below the edge count.

In [3]:
positions, parent, swc_type, radius = neurons[0]
edges = np.column_stack([np.arange(1, len(positions)), parent[1:]])

summary = write_graph(
    STORE,
    positions=positions,
    edges=edges,
    chunk_shape=(100.0, 100.0, 100.0),
    bin_shape=(25.0, 25.0, 25.0),
    kind="skeleton",
    vertex_attributes={"radius": radius, "swc_type": swc_type},
)
print(summary)

result = read_graph(STORE)
print(f"\nread_graph: {result['node_count']:,} nodes, {result['edge_count']:,} edges")
ds = zv.open(STORE)
full = ds.read()
print(f"{ds.kinds[0]}: {ds.level(0).objects.count} object, attributes {full.attributes.names()}")
types, counts = np.unique(full.attributes["swc_type"], return_counts=True)
print("nodes per SWC type:", dict(zip(types.astype(int).tolist(), counts.tolist())))

{'node_count': 1618, 'edge_count': 1617, 'chunk_count': 11, 'intra_edge_count': 33, 'cross_edge_count': 63, 'object_count': 1, 'kind': 'skeleton'}

read_graph: 1,618 nodes, 1,617 edges
skeleton: 1 object, attributes ('radius', 'swc_type')
nodes per SWC type: {1: 1, 2: 412, 3: 1205}


## 3 · Several skeletons in one store

`object_ids` gives each vertex its tree; edges use indices into the
concatenated arrays. `read_graph` reads the whole level; it has no object
filter for graphs and skeletons. Per-tree results come from the skeleton
metrics (section 5) or a per-object SWC export (section 4).

In [4]:
offsets = np.cumsum([0] + [len(n[0]) for n in neurons[:-1]])
all_positions = np.concatenate([n[0] for n in neurons])
all_edges = np.concatenate([
    np.column_stack([np.arange(1, len(n[0])), n[1][1:]]) + off
    for n, off in zip(neurons, offsets)
])
object_ids = np.concatenate([np.full(len(n[0]), i) for i, n in enumerate(neurons)])

summary = write_graph(
    STORE_MULTI,
    positions=all_positions,
    edges=all_edges,
    chunk_shape=(100.0, 100.0, 100.0),
    bin_shape=(25.0, 25.0, 25.0),
    kind="skeleton",
    object_ids=object_ids,
    vertex_attributes={"radius": np.concatenate([n[3] for n in neurons])},
)
print(summary)
print(f"objects: {zv.open(STORE_MULTI).level(0).objects.count}")

{'node_count': 4816, 'edge_count': 4813, 'chunk_count': 24, 'intra_edge_count': 103, 'cross_edge_count': 188, 'object_count': 3, 'kind': 'skeleton'}
objects: 3


## 4 · SWC round trip

`ingest_swc` makes one object per tree. Objects are numbered from 0 in the
order the roots appear, and `object_attributes/segment_id` numbers the same
trees from 1. `radius` and `compartment` (the SWC type) become vertex
attributes. `export_swc` writes one file for the level, or, with `object_ids`,
a directory with one file per tree, named by segment id.

In [5]:
from zarr_vectors_tools.convert.export.swc import export_swc
from zarr_vectors_tools.convert.ingest.swc import ingest_swc

with open(SWC_IN, "w") as f:
    f.write("# synthetic neurons\n")
    base = 0
    for pos, parent, swc_type, radius in neurons:
        for i in range(len(pos)):
            p = -1 if parent[i] < 0 else int(parent[i]) + base + 1
            f.write(f"{base + i + 1} {swc_type[i]} {pos[i, 0]:.3f} {pos[i, 1]:.3f} "
                    f"{pos[i, 2]:.3f} {radius[i]:.3f} {p}\n")
        base += len(pos)

summary = ingest_swc(SWC_IN, STORE_SWC, (100.0, 100.0, 100.0))
print(summary)

from zarr_vectors.building import get_resolution_level, open_store, read_object_attributes

level0 = zv.open(STORE_SWC).level(0)
print(f"vertex attrs: {level0.attribute_names('vertex')}")
seg = read_object_attributes(get_resolution_level(open_store(STORE_SWC), 0), "segment_id")
print(f"segment_id per object: {seg.tolist()}")

swc_out = os.path.join(_tmpdir, "roundtrip.swc")
print(export_swc(STORE_SWC, swc_out))
rows = [line.split() for line in open(swc_out) if not line.startswith("#")]
print(f"exported rows: {len(rows):,} (input {sum(len(n[0]) for n in neurons):,}), "
      f"roots: {sum(r[-1] == '-1' for r in rows)}")

per_tree = os.path.join(_tmpdir, "per_tree")
export_swc(STORE_SWC, per_tree, object_ids=[0, 2])
print(f"per-tree files: {sorted(os.listdir(per_tree))}")

{'node_count': 4816, 'edge_count': 4813, 'chunk_count': 24, 'intra_edge_count': 103, 'cross_edge_count': 188, 'object_count': 3, 'kind': 'skeleton'}
vertex attrs: ('compartment', 'radius')
segment_id per object: [1, 2, 3]
{'node_count': 4816, 'root_count': 3, 'attributes_carried': ['radius', 'compartment']}
exported rows: 4,816 (input 4,816), roots: 3


per-tree files: ['1.swc', '3.swc']


## 5 · Pyramid and per-tree metrics

The skeleton coarsener keeps every 4th vertex along each branch (coarsen
factor 4) and always keeps roots, branch points, tips and the vertices where
a branch crosses a chunk boundary, so each tree stays one tree. Doubling the
chunk edge per level (`chunk_scale_factors`) leaves fewer boundary vertices to
keep. Sparsity 1 keeps every tree.

`compute_skeleton_metrics` measures each tree at one level; `write=False`
leaves the store unchanged.

In [6]:
from zarr_vectors_tools.algorithms import compute_skeleton_metrics
from zarr_vectors_tools.multiresolution.coarsen import build_pyramid

build_pyramid(STORE_SWC, factors=[(4, 1), (4, 1)], chunk_scale_factors=[2, 2])

ds = zv.open(STORE_SWC)
columns = ["node_count", "component_count", "leaf_count", "branch_count", "cable_length"]
for lv in ds.levels:
    level = ds.level(lv)
    print(f"Level {lv}: {level.vertex_count:,} vertices, {level.objects.count} trees, "
          f"chunk {level.scale[0]:.0f} µm")
    metrics = compute_skeleton_metrics(STORE_SWC, level=lv, write=False)
    print(metrics[columns].round(1).to_string(), end="\n\n")

Level 0: 4,816 vertices, 3 trees, chunk 100 µm
           node_count  component_count  leaf_count  branch_count  cable_length
object_id                                                                     
0                1618                1          38            36        3234.0
1                1550                1          40            38        3098.0
2                1648                1          38            36        3294.0

Level 1: 1,322 vertices, 3 trees, chunk 200 µm
           node_count  component_count  leaf_count  branch_count  cable_length
object_id                                                                     
0                 452                1          38            36        3056.9
1                 424                1          40            38        2935.5
2                 446                1          38            36        3120.2

Level 2: 431 vertices, 3 trees, chunk 400 µm
           node_count  component_count  leaf_count  branch_count  cab

Every level keeps three one-component trees with the same tips and branch
points, with 3–3.6× fewer vertices per level. Cable length shrinks at
each level, about 5% at level 1 and 18% at level 2 here, as the straight edges
between kept vertices cut the corners of each branch.

## 6 · Validate

Level 4 checks tree topology; level 5 adds the pyramid.

In [7]:
print(validate(STORE_MULTI, level=4).summary())
print(validate(STORE_SWC, level=5).summary())

Level 4 validation: PASS
  26 passed, 0 warnings, 0 errors


Level 5 validation: PASS
  55 passed, 0 warnings, 0 errors


## Summary

| Step | API |
|------|-----|
| Write | `write_graph(path, positions, edges, kind="skeleton", object_ids=..., vertex_attributes=...)` |
| Read | `read_graph(path)` → `positions`, `edges`; `zarr_vectors.open(path).read()` adds attributes |
| SWC ingest | `ingest_swc(swc_path, store_path, chunk_shape)` |
| SWC export | `export_swc(store_path, swc_path)`; `object_ids=[...]` writes one file per tree |
| Pyramid | `build_pyramid(path, factors=[(4, 1), (4, 1)], chunk_scale_factors=[2, 2])` |
| Per-tree metrics | `compute_skeleton_metrics(path, level=n, write=False)` |

Graph algorithms (components, search, communities) also run on skeleton
stores; see notebooks 07, 08 and 10.